# EUSoft Customer Satisfaction Classification (NLP)

Objective: Train an AI model to classify whether a customer review indicates satisfaction.
Approach: Fine-tune a transformer model (RoBERTa) on labeled sentiment data (positive/negative), evaluate accuracy, then predict sentiment for 20 EUSoft customer reviews.

In [1]:
!pip -q install transformers datasets evaluate accelerate openpyxl

In [2]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: You are on CPU. In Colab: Runtime -> Change runtime type -> GPU")


Using device: cpu


## Load Train/Test Data + Clean Labels

We load `Train.csv` and `Test.csv`.
The `sentiment` column is standardized (lowercase + trimmed) to prevent mapping errors (e.g., "Positive" vs "positive").
Then we map labels: positive → 1 and negative → 0, and rename the text column to `text` for downstream tokenization.

In [ ]:
df_train = pd.read_csv("Train.csv")
df_test  = pd.read_csv("Test.csv")

df_train["sentiment"] = df_train["sentiment"].astype(str).str.lower().str.strip()
df_test["sentiment"]  = df_test["sentiment"].astype(str).str.lower().str.strip()

print("Unique labels in train:", df_train["sentiment"].unique())

label_mapping = {"positive": 1, "negative": 0}
df_train["label"] = df_train["sentiment"].map(label_mapping)
df_test["label"]  = df_test["sentiment"].map(label_mapping)

df_train = df_train.rename(columns={"review": "text"})
df_test  = df_test.rename(columns={"review": "text"})

df_train = df_train.dropna(subset=["text", "label"])
df_test  = df_test.dropna(subset=["text", "label"])

df_train["label"] = df_train["label"].astype(int)
df_test["label"]  = df_test["label"].astype(int)

print("Training samples:", len(df_train))
print("Testing samples:", len(df_test))

## Load EUSoft Reviews (20 Customers)

This file contains 20 customer reviews from EUSoft’s CRM clients.
These reviews are the real target data we want to classify as satisfied/unsatisfied (predicted sentiment).

In [ ]:
eusoft_filename = "EUSoft_Reviews.xlsx"
if os.path.exists(eusoft_filename):
    eusoft_df = pd.read_excel(eusoft_filename)
    print("Loaded:", eusoft_filename, "rows:", len(eusoft_df))
else:
    raise FileNotFoundError("Please upload EUSoft_Reviews.xlsx next to this notebook.")

## Tokenization (Preparing Text for RoBERTa)

Transformers cannot directly process raw text.
We use the RoBERTa tokenizer to convert each review into token IDs and attention masks.
We truncate long reviews and cap the sequence length at 512 tokens.

In [ ]:
BASE_MODEL = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

dataset_train = Dataset.from_pandas(df_train)
dataset_test  = Dataset.from_pandas(df_test)

print("Tokenizing data...")
tokenized_train = dataset_train.map(preprocess_function, batched=True)
tokenized_test  = dataset_test.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Model Training (Supervised Learning)

We use supervised learning because the training data contains ground-truth labels (positive/negative).
We fine-tune a pre-trained RoBERTa model for binary sequence classification.
Training is set to 5 epochs as required by the assignment.

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="EUSoft_Model_Output",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
print("Starting training...")
trainer.train()

In [ ]:
print("Evaluating final model...")
eval_metrics = trainer.evaluate()
eval_metrics

## Predict Sentiment for the 20 EUSoft Reviews

We locate the review text column (usually named `Review`).
For each review, we tokenize the text, run the model forward pass, and select the class with the highest logit.
Results are saved to a CSV file for reporting.

In [ ]:
model.to(device)

review_col = "Review" if "Review" in eusoft_df.columns else "review"
if review_col not in eusoft_df.columns:
    if len(eusoft_df.columns) > 1:
        review_col = eusoft_df.columns[1]
    else:
        raise ValueError("No review column found.")

def predict_sentiment(text):
    if not isinstance(text, str):
        return "Unknown"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_class_id = logits.argmax().item()
    return id2label[predicted_class_id]

eusoft_df["Sentiment"] = eusoft_df[review_col].apply(predict_sentiment)

output_file = "EUSoft_Reviews_Predicted_Final.csv"
eusoft_df.to_csv(output_file, index=False)
print("Saved:", output_file)

eusoft_df[[review_col, "Sentiment"]].head()